# Transcript Q&A attribution — who said that, and when?

Meeting notes and transcript Q&A are easier to navigate when each generated
statement links to the relevant moment. Instead of searching a full recording,
the reader can jump to **who** said it, **when** in the call, and in **what
words**.

[TokenPath](https://tokenpath.ai) adds source navigation to spans in a summary
or answer. The transcript is the document;
attribution returns character offsets into it; a tiny lookup maps offsets →
speaker + timestamp. Result: each linked bullet in your meeting notes carries a
receipt like *— Maya, 00:01:26*, and clicking it jumps to the moment in the
transcript (or, with your player, the moment in the recording).

No changes to how you generate the notes — this runs *after* your existing
summarizer, on any model's output.

## Setup

You need a TokenPath API key — free at [platform.tokenpath.ai](https://platform.tokenpath.ai)
(10M attributed tokens on signup, no card required). Export it as `TOKENPATH_API_KEY`
before running this notebook.

In [1]:
%pip install -q requests

In [2]:
import os
import re

import requests

API_URL = "https://api.tokenpath.ai"
API_KEY = os.environ.get("TOKENPATH_API_KEY")
assert API_KEY, (
    "Set TOKENPATH_API_KEY first — grab a free key (10M tokens, no card) "
    "at https://platform.tokenpath.ai"
)


def attribute(document, question, answer, spans, **options):
    """Resolve each answer character span to the source span that produced it."""
    response = requests.post(
        f"{API_URL}/v1/attributions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "document": document,
            "question": question,
            "answer": answer,
            "spans": spans,
            **options,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

## A meeting transcript

Speaker-labeled and timestamped — the shape every meeting tool already has.

In [3]:
TRANSCRIPT = """\
[00:00:04] Maya: Okay, quick sync on the March launch. Dev, where are we on billing?
[00:00:19] Dev: Honestly, the billing migration slipped. Usage-based proration turned out to be way hairier than we scoped. I need two more weeks to do it right.
[00:00:41] Maya: Two weeks puts us past March 3rd. Can we cut scope instead?
[00:00:52] Dev: We could ship without annual plans, but proration has to work on day one or we'll be issuing refunds by hand.
[00:01:10] Sara: From my side, the new onboarding flow is on track. Designs are done, I'm handing them to Dev's team by February 20th.
[00:01:26] Maya: Good. Then let's make the call: launch moves to March 17th. Billing done right beats billing on time.
[00:01:41] Tom: Fine by sales, but I need the pricing locked this week. Prospects keep asking and I have nothing to send them.
[00:01:55] Maya: It's locked: $49 per seat per month, 14-day free trial, no credit card up front. Annual pricing follows when Dev ships annual plans.
[00:02:14] Tom: And the Meridian deal? They want a pilot before they'll sign anything.
[00:02:24] Maya: Give Meridian a 60-day pilot at half price, and get it in writing that they're a reference customer if we hit the success criteria.
[00:02:41] Dev: One more thing — the SOC 2 audit. The auditors need our access-review evidence by Friday or we lose the March window entirely.
[00:02:55] Maya: That's yours and mine, we'll do it Thursday morning. Anything else? Okay — Sara sends designs by the 20th, Dev owns proration, Tom gets pricing today.
"""

## Index the transcript

Parse each utterance's character range once. After attribution, mapping a
source span to *speaker + timestamp* is a lookup, not a search: find the
utterances that overlap the span, keep the one with the most overlap.

In [4]:
UTTERANCE = re.compile(r"^\[(\d\d:\d\d:\d\d)\] (\w+): (.*)$", re.MULTILINE)

utterances = [
    {
        "timestamp": m.group(1),
        "speaker": m.group(2),
        "text": m.group(3),
        "start": m.start(),
        "end": m.end(),
    }
    for m in UTTERANCE.finditer(TRANSCRIPT)
]


def locate(source_start, source_end):
    """Map a source char span to the utterance that carries most of it."""
    best, best_overlap = None, 0
    for u in utterances:
        overlap = min(source_end, u["end"]) - max(source_start, u["start"])
        if overlap > best_overlap:
            best, best_overlap = u, overlap
    return best


print(f"{len(utterances)} utterances,",
      f"{utterances[0]['speaker']} opens at {utterances[0]['timestamp']}")

12 utterances, Maya opens at 00:00:04


## Meeting notes with receipts

Here are the AI-generated notes for this call (from your existing summarizer —
any model). Each bullet is one claim; attribute them all in one call and
resolve each to its speaker and timestamp.

In [5]:
NOTES = """\
Launch moved from March 3rd to March 17th because the billing migration slipped.
Pricing locked at $49 per seat per month with a 14-day free trial and no credit card required.
Meridian gets a 60-day pilot at half price in exchange for being a reference customer.
Sara delivers onboarding designs to engineering by February 20th.
SOC 2 access-review evidence is due Friday; Maya and Dev will prepare it Thursday morning."""

QUESTION = "Summarize the key decisions and action items from this meeting."

bullets = []
cursor = 0
for line in NOTES.split("\n"):
    bullets.append([cursor, cursor + len(line)])
    cursor += len(line) + 1

result = attribute(TRANSCRIPT, QUESTION, NOTES, bullets)

for item in result["spans"]:
    source = item["source"]
    u = locate(source["start"], source["end"]) if source else None
    receipt = f"{u['speaker']}, {u['timestamp']}" if u else "citation unavailable"
    print(f"* {item['answer']['text']}")
    print(f"    -- {receipt}  (confidence {source['confidence']:.2f})" if source
          else f"    -- {receipt}")
    if u:
        print(f"       \"{u['text'][:80]}…\"" if len(u["text"]) > 80
              else f"       \"{u['text']}\"")
    print()

* Launch moved from March 3rd to March 17th because the billing migration slipped.
    -- Maya, 00:01:26  (confidence 0.94)
       "Good. Then let's make the call: launch moves to March 17th. Billing done right b…"

* Pricing locked at $49 per seat per month with a 14-day free trial and no credit card required.
    -- Maya, 00:01:55  (confidence 1.00)
       "It's locked: $49 per seat per month, 14-day free trial, no credit card up front.…"

* Meridian gets a 60-day pilot at half price in exchange for being a reference customer.
    -- Maya, 00:02:24  (confidence 0.99)
       "Give Meridian a 60-day pilot at half price, and get it in writing that they're a…"

* Sara delivers onboarding designs to engineering by February 20th.
    -- Sara, 00:01:10  (confidence 0.97)
       "From my side, the new onboarding flow is on track. Designs are done, I'm handing…"

* SOC 2 access-review evidence is due Friday; Maya and Dev will prepare it Thursday morning.
    -- Maya, 00:02:55  (confidence 0.9

## Q&A over the transcript, same mechanics

Answers get the same treatment as notes: split into claims, attribute, point
back into the conversation.

In [6]:
def claim_spans(text):
    """Sentence-level [start, end) character spans over `text`."""
    boundary = re.compile(r"[.!?][\"\')\]]*(?=\s|$)|\n")
    raw, start = [], 0
    for m in boundary.finditer(text):
        raw.append((start, m.end()))
        start = m.end()
    if start < len(text):
        raw.append((start, len(text)))
    spans = []
    for s, e in raw:
        segment = text[s:e]
        if segment.strip():
            left = len(segment) - len(segment.lstrip())
            right = len(segment) - len(segment.rstrip())
            spans.append([s + left, e - right])
    return spans


qa_question = "Why did the launch date move?"
qa_answer = (
    "The launch moved to March 17th because the billing migration slipped — "
    "usage-based proration was harder than scoped, and Dev needed two more "
    "weeks to do it right rather than issue refunds by hand."
)

qa = attribute(TRANSCRIPT, qa_question, qa_answer, claim_spans(qa_answer))
for item in qa["spans"]:
    source = item["source"]
    u = locate(source["start"], source["end"]) if source else None
    print(f"{item['answer']['text']}")
    print(f"    -- {u['speaker']}, {u['timestamp']}: \"{u['text']}\"\n" if u
          else "    -- citation unavailable\n")

The launch moved to March 17th because the billing migration slipped — usage-based proration was harder than scoped, and Dev needed two more weeks to do it right rather than issue refunds by hand.
    -- Dev, 00:00:19: "Honestly, the billing migration slipped. Usage-based proration turned out to be way hairier than we scoped. I need two more weeks to do it right."



## Render it: notes your users can click

The same data as a self-contained HTML page — notes on top, transcript below,
each receipt scrolls to and flashes the utterance it came from. This is the
piece you'd embed in a meeting-notes UI (and if you have the recording,
`timestamp` is your seek target).

In [7]:
import html as html_lib

from IPython.display import HTML, display


def notes_page(result):
    note_items = []
    for i, item in enumerate(result["spans"]):
        source = item["source"]
        u = locate(source["start"], source["end"]) if source else None
        text = html_lib.escape(item["answer"]["text"])
        if u:
            idx = utterances.index(u)
            note_items.append(
                f"<li>{text} <a class='receipt' href='#u-{idx}'>"
                f"{u['speaker']}, {u['timestamp']}</a></li>"
            )
        else:
            note_items.append(
                f"<li>{text} <span class='receipt'>citation unavailable</span></li>"
            )

    transcript_lines = [
        f"<p id='u-{i}'><b>[{u['timestamp']}] {u['speaker']}:</b> "
        f"{html_lib.escape(u['text'])}</p>"
        for i, u in enumerate(utterances)
    ]

    return f"""<div style="font-family:Menlo,Consolas,monospace;font-size:13px;
line-height:1.8;max-width:720px">
<style>
  .receipt {{ background:#e6dcf7; color:#1a1a1a; border-radius:3px;
              padding:0 5px; text-decoration:none; font-size:11px; }}
  p:target {{ background:#e6dcf7; border-radius:3px; }}
</style>
<h3>Meeting notes</h3><ul>{"".join(note_items)}</ul>
<h3>Transcript</h3>{"".join(transcript_lines)}
</div>"""


page = notes_page(result)
display(HTML(page))

with open("meeting-notes.html", "w") as f:
    f.write("<!doctype html><meta charset='utf-8'>" + page)
print("also wrote meeting-notes.html")

also wrote meeting-notes.html


## Production notes

- **Long calls**: an hour of speech is roughly 9–10k words — well inside the
  400k-character document cap, so a full call is one attribution call. For
  multi-hour recordings, attribute per chapter and merge.
- **Timestamps are seek targets**: `locate()` gives you the utterance; your
  player gives you the jump-to-moment. That's the demo that sells — click a
  bullet, hear the person say it.
- **Diarization quality matters**: receipts inherit the transcript's speaker
  labels. Attribution is offset-based, so it stays correct even if labels are
  wrong — it points at the utterance; the label is your transcript's claim.
- **Works on any summarizer's output** — the notes above could come from your
  current pipeline unchanged; attribution is a post-processing call.

---

## Where to go next

- **API reference & guides** — [docs.tokenpath.ai](https://docs.tokenpath.ai)
- **Bugs / broken examples** — [open an issue](https://github.com/tokenpath/tokenpath-cookbook/issues)
- **"How do I…?"** — [start a discussion](https://github.com/tokenpath/tokenpath-cookbook/discussions)
- **Email** — support@tokenpath.ai